# Pydantic V2 深度进阶

## 学习重点
不要只用 Pydantic 声明简单的字段类型。要深入掌握：
1. 自定义校验器（`@field_validator`）
2. 复杂嵌套结构（Nested Models）
3. 动态 Schema 生成

## 核心痛点
大模型有时会把日期格式传错、把字符串类型的 ID 传成数字，你必须在 Pydantic 层做强行拦截和类型清洗。

## 0. 环境准备

### 0.1 用 uv 创建虚拟环境

虚拟环境放在项目根目录下，所有课程共享同一个 venv，避免重复创建。

在终端中执行以下命令（**不要在 Notebook 中运行**）：

```bash
# 进入项目根目录
cd ~/Documents/ObsidianVault/CS/KnowledgeBase

# 如果根目录下还没有 .venv，则创建（已有则跳过）
uv venv .venv

# 激活虚拟环境
source .venv/bin/activate

# 安装本课程所需依赖
uv pip install pydantic ipykernel

# 将虚拟环境注册为 Jupyter Kernel
python -m ipykernel install --user --name=knowledgebase --display-name="KnowledgeBase (.venv)"
```

然后在 Jupyter Notebook 中选择 Kernel 为 **KnowledgeBase (.venv)** 即可。

> **提示**：后续其他课程如需额外依赖，只需 `uv pip install <package>` 即可，无需重新创建 venv。

### 0.2 验证环境

In [1]:
import sys
print(f"Python: {sys.version}")
print(f"Python 路径: {sys.executable}")

import pydantic
print(f"Pydantic: {pydantic.__version__}")

Python: 3.13.3 (main, Apr  8 2025, 13:54:08) [Clang 15.0.0 (clang-1500.1.0.2.5)]
Python 路径: /Users/savior/Documents/ObsidianVault/CS/KnowledgeBase/.venv/bin/python
Pydantic: 2.13.4


### 0.3 导入依赖

In [2]:
from pydantic import BaseModel, Field, field_validator, model_validator, ConfigDict
from pydantic import ValidationError
from typing import Optional, Literal, Annotated
from datetime import datetime, date
import json

---
## 1. @field_validator — 字段级校验与类型清洗

这是 Pydantic V2 中最常用的校验机制，用于对**单个字段**做自定义校验和类型转换，默认值 `mode="after"` 在 Pydantic 执行任何内置的类型检查或转换之后调用，关注的就是**业务逻辑校验**。

### 1.1 基本用法

In [ ]:
class User(BaseModel):
    name: str
    age: int

    @field_validator('name')
    @classmethod
    def name_must_not_be_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('name 不能为空')
        return v.strip()

    @field_validator('age')
    @classmethod
    def age_must_be_positive(cls, v: int) -> int:
        if v < 0 or v > 150:
            raise ValueError(f'age 必须在 0~150 之间，收到 {v}')
        return v

user = User(name="  Alice  ", age=30)
print(f"name: '{user.name}'")
print(f"age: {user.age}")

### 1.2 核心痛点：大模型传错类型 — 类型清洗

大模型经常把字符串 ID 传成数字，或者把日期格式传错。我们用 `@field_validator` 在入口处做强制清洗。此处加上 `mode="before"` 在 Pydantic 执行任何内置的类型检查或转换之前调用，关注的就是**格式和类型问题**。

In [4]:
class ToolInput(BaseModel):
    user_id: str
    order_date: date

    @field_validator('user_id', mode='before')
    @classmethod
    def coerce_user_id_to_str(cls, v):
        if isinstance(v, int):
            return str(v)
        if isinstance(v, str) and v.isdigit():
            return v
        raise ValueError(f'user_id 必须是数字字符串，收到 {v!r}')

    @field_validator('order_date', mode='before')
    @classmethod
    def parse_flexible_date(cls, v):
        if isinstance(v, date):
            return v
        if isinstance(v, str):
            for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%d-%m-%Y', '%Y年%m月%d日'):
                try:
                    return datetime.strptime(v, fmt).date()
                except ValueError:
                    continue
            raise ValueError(f'无法解析日期: {v!r}，支持的格式: YYYY-MM-DD, YYYY/MM/DD, DD-MM-YYYY, YYYY年MM月DD日')
        raise ValueError(f'不支持的日期类型: {type(v)}')

ok1 = ToolInput(user_id=12345, order_date="2024-01-15")
print(f"ok1: user_id={ok1.user_id!r}, order_date={ok1.order_date}")

ok2 = ToolInput(user_id="67890", order_date="2024/03/20")
print(f"ok2: user_id={ok2.user_id!r}, order_date={ok2.order_date}")

ok3 = ToolInput(user_id=42, order_date="15-01-2024")
print(f"ok3: user_id={ok3.user_id!r}, order_date={ok3.order_date}")

ok4 = ToolInput(user_id="99", order_date="2024年12月25日")
print(f"ok4: user_id={ok4.user_id!r}, order_date={ok4.order_date}")

ok1: user_id='12345', order_date=2024-01-15
ok2: user_id='67890', order_date=2024-03-20
ok3: user_id='42', order_date=2024-01-15
ok4: user_id='99', order_date=2024-12-25


In [5]:
try:
    ToolInput(user_id="abc", order_date="2024-01-15")
except ValidationError as e:
    print(f"校验失败: {e.errors()[0]['msg']}")

try:
    ToolInput(user_id=123, order_date="Jan 15 2024")
except ValidationError as e:
    print(f"校验失败: {e.errors()[0]['msg']}")

校验失败: Value error, user_id 必须是数字字符串，收到 'abc'
校验失败: Value error, 无法解析日期: 'Jan 15 2024'，支持的格式: YYYY-MM-DD, YYYY/MM/DD, DD-MM-YYYY, YYYY年MM月DD日


### 1.3 mode 参数详解

| mode | 执行时机 | 用途 |
|------|---------|------|
| `before` | Pydantic 类型转换**之前** | 类型清洗、格式预处理 |
| `after` | Pydantic 类型转换**之后**（默认） | 对已转换的值做业务校验 |
| `wrap` | 包裹默认校验逻辑 | 需要同时控制前后处理时使用 |

In [6]:
class Demo(BaseModel):
    value: int

    @field_validator('value', mode='before')
    @classmethod
    def before_coerce(cls, v):
        print(f"  [before] 原始值: {v!r} (type={type(v).__name__})")
        if isinstance(v, str) and 'pi' in v.lower():
            return 3
        return v

    @field_validator('value', mode='after')
    @classmethod
    def after_check(cls, v):
        print(f"  [after] 转换后: {v!r} (type={type(v).__name__})")
        if v > 100:
            raise ValueError(f'value 不能超过 100')
        return v

print("--- 测试 'pi' 字符串 ---")
d1 = Demo(value="pi is 3.14")
print(f"结果: {d1.value}")

print("\n--- 测试普通字符串 ---")
d2 = Demo(value="42")
print(f"结果: {d2.value}")

--- 测试 'pi' 字符串 ---
  [before] 原始值: 'pi is 3.14' (type=str)
  [after] 转换后: 3 (type=int)
结果: 3

--- 测试普通字符串 ---
  [before] 原始值: '42' (type=str)
  [after] 转换后: 42 (type=int)
结果: 42


In [7]:
class User(BaseModel):
    age: int

    @field_validator('age', mode='wrap')
    @classmethod
    def fallback_age(cls, v, handler):
        try:
            # 尝试让 Pydantic 正常解析（比如把 "23" 转成 23）
            return handler(v)
        except (ValueError, TypeError):
            # 如果解析失败（比如传入了 "abc"），不报错，而是降级返回 0
            return 0

# 测试
print(User(age="23"))  # 正常解析: age=23
print(User(age="abc")) # 遇到错误被 wrap 拦截并降级: age=0

age=23
age=0


wrap主要是做错误拦截/降级  和 条件跳过

wrap 既利用了 Pydantic 强大的解析能力（handler(v)），又拿到了最终的控制权。

在 wrap 中，如果你不调用 handler(v)，Pydantic 的后续校验大门就直接关闭了。这就实现了真正的、彻底的条件跳过。

### 1.4 方法是类方法

因为方法校验器必须在类对象被示例化之前运行

---
## 2. @model_validator — 跨字段校验

当校验逻辑涉及**多个字段之间的关系**时，用 `@model_validator`。

In [8]:
class DateRange(BaseModel):
    start_date: date
    end_date: date

    @model_validator(mode='after')
    def check_date_order(self):
        if self.start_date > self.end_date:
            raise ValueError(
                f'start_date ({self.start_date}) 不能晚于 end_date ({self.end_date})'
            )
        return self

ok = DateRange(start_date="2024-01-01", end_date="2024-12-31")
print(f"合法: {ok.start_date} ~ {ok.end_date}")

try:
    DateRange(start_date="2024-12-31", end_date="2024-01-01")
except ValidationError as e:
    print(f"非法: {e.errors()[0]['msg']}")

合法: 2024-01-01 ~ 2024-12-31
非法: Value error, start_date (2024-12-31) 不能晚于 end_date (2024-01-01)


In [ ]:
class SearchQuery(BaseModel):
    query: str
    limit: int = 10
    offset: int = 0

    @field_validator('limit')
    @classmethod
    def limit_range(cls, v):
        if v < 1 or v > 100:
            raise ValueError(f'limit 必须在 1~100 之间，收到 {v}')
        return v

    @model_validator(mode='after')
    def check_offset_with_limit(self):
        if self.offset >= 1000:
            raise ValueError(f'offset 过大 ({self.offset})，请使用更精确的查询条件')
        return self

ok = SearchQuery(query="pydantic", limit=20, offset=50)
print(f"合法: query={ok.query}, limit={ok.limit}, offset={ok.offset}")

try:
    SearchQuery(query="test", limit=20, offset=2000)
except ValidationError as e:
    print(f"非法: {e.errors()[0]['msg']}")

---
## 3. 复杂嵌套结构（Nested Models）

真实场景中，工具参数往往是多层嵌套的 JSON 结构。Pydantic 支持模型嵌套，并自动递归校验。

In [9]:
class Address(BaseModel):
    city: str
    street: str
    zip_code: str

    @field_validator('zip_code')
    @classmethod
    def zip_code_format(cls, v):
        import re
        if not re.match(r'^\d{6}$', v):
            raise ValueError(f'zip_code 必须是6位数字，收到 {v!r}')
        return v

class Company(BaseModel):
    name: str
    industry: str
    address: Address

class UserProfile(BaseModel):
    name: str
    age: int
    email: str
    company: Optional[Company] = None

    @field_validator('email')
    @classmethod
    def email_format(cls, v):
        if '@' not in v:
            raise ValueError(f'email 格式不合法: {v!r}')
        return v

data = {
    "name": "Alice",
    "age": 30,
    "email": "alice@example.com",
    "company": {
        "name": "TechCorp",
        "industry": "AI",
        "address": {
            "city": "Beijing",
            "street": "Zhongguancun St",
            "zip_code": "100080"
        }
    }
}

profile = UserProfile(**data)
print(f"name: {profile.name}")
print(f"company: {profile.company.name}")
print(f"city: {profile.company.address.city}")
print(f"zip_code: {profile.company.address.zip_code}")

name: Alice
company: TechCorp
city: Beijing
zip_code: 100080


In [10]:
try:
    UserProfile(
        name="Bob",
        age=25,
        email="bob-example.com",
        company={
            "name": "Corp",
            "industry": "Finance",
            "address": {
                "city": "Shanghai",
                "street": "Nanjing Rd",
                "zip_code": "ABC123"
            }
        }
    )
except ValidationError as e:
    for err in e.errors():
        loc = ' -> '.join(str(x) for x in err['loc'])
        print(f"路径: {loc} | 错误: {err['msg']}")

路径: email | 错误: Value error, email 格式不合法: 'bob-example.com'
路径: company -> address -> zip_code | 错误: Value error, zip_code 必须是6位数字，收到 'ABC123'


### 3.1 列表嵌套

In [ ]:
class SearchResult(BaseModel):
    title: str
    url: str
    relevance_score: float = Field(ge=0, le=1)

class SearchResponse(BaseModel):
    query: str
    results: list[SearchResult]
    total: int

    @model_validator(mode='after')
    def total_matches_results(self):
        if self.total != len(self.results):
            raise ValueError(
                f'total ({self.total}) 与 results 数量 ({len(self.results)}) 不一致'
            )
        return self

resp = SearchResponse(
    query="pydantic",
    results=[
        {"title": "Pydantic Docs", "url": "https://docs.pydantic.dev", "relevance_score": 0.95},
        {"title": "Pydantic GitHub", "url": "https://github.com/pydantic/pydantic", "relevance_score": 0.88},
    ],
    total=2
)

for r in resp.results:
    print(f"[{r.relevance_score:.2f}] {r.title} - {r.url}")

---
## 4. 动态 Schema 生成

在 LLM 工具调用场景中，需要把 Pydantic Model 转成 JSON Schema 传给大模型。Pydantic V2 提供了强大的 Schema 生成能力。

In [11]:
class WeatherQuery(BaseModel):
    """查询指定城市的天气信息"""
    city: str = Field(description="城市名称，如 '北京', '上海'")
    date: str = Field(description="查询日期，格式 YYYY-MM-DD")
    unit: Literal["celsius", "fahrenheit"] = Field(default="celsius", description="温度单位")

    @field_validator('date', mode='before')
    @classmethod
    def validate_date(cls, v):
        try:
            datetime.strptime(v, '%Y-%m-%d')
        except ValueError:
            raise ValueError(f'日期格式错误: {v!r}，需要 YYYY-MM-DD')
        return v

schema = WeatherQuery.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

{
  "description": "查询指定城市的天气信息",
  "properties": {
    "city": {
      "description": "城市名称，如 '北京', '上海'",
      "title": "City",
      "type": "string"
    },
    "date": {
      "description": "查询日期，格式 YYYY-MM-DD",
      "title": "Date",
      "type": "string"
    },
    "unit": {
      "default": "celsius",
      "description": "温度单位",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "title": "Unit",
      "type": "string"
    }
  },
  "required": [
    "city",
    "date"
  ],
  "title": "WeatherQuery",
  "type": "object"
}


### 4.1 生成 OpenAI Function Calling 格式

In [ ]:
def pydantic_to_openai_tool(model_cls: type[BaseModel]) -> dict:
    """将 Pydantic Model 转换为 OpenAI Function Calling 的 tools 格式"""
    schema = model_cls.model_json_schema()
    return {
        "type": "function",
        "function": {
            "name": model_cls.__name__,
            "description": model_cls.__doc__ or "",
            "parameters": schema
        }
    }

tool_def = pydantic_to_openai_tool(WeatherQuery)
print(json.dumps(tool_def, indent=2, ensure_ascii=False))

### 4.2 动态创建 Model

有时工具参数需要根据运行时配置动态生成，可以用 `create_model`。

In [ ]:
from pydantic import create_model

def build_db_query_model(allowed_tables: list[str]):
    table_literal = Literal[tuple(allowed_tables)]
    return create_model(
        'DBQuery',
        table=(table_literal, Field(description=f"目标表，可选: {', '.join(allowed_tables)}")),
        conditions=(Optional[str], Field(default=None, description='WHERE 条件')),
        limit=(int, Field(default=10, ge=1, le=1000, description='返回行数上限')),
    )

DBQuery = build_db_query_model(['users', 'orders', 'products'])

q1 = DBQuery(table='users', limit=50)
print(f"q1: table={q1.table}, conditions={q1.conditions}, limit={q1.limit}")

schema = DBQuery.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

In [ ]:
try:
    DBQuery(table='secrets', limit=10)
except ValidationError as e:
    print(f"非法表名: {e.errors()[0]['msg']}")

---
## 5. 实战：LLM 工具调用的防御性校验

将以上所有技术组合起来，构建一个生产级的工具输入校验模型。

In [ ]:
class APIEndpoint(BaseModel):
    path: str = Field(description="API 路径，如 /api/v1/users")
    method: Literal["GET", "POST", "PUT", "DELETE"] = Field(description="HTTP 方法")

    @field_validator('path', mode='before')
    @classmethod
    def normalize_path(cls, v):
        if isinstance(v, str):
            v = v.strip()
            if not v.startswith('/'):
                v = '/' + v
            return v
        raise ValueError(f'path 必须是字符串')

class APIRequest(BaseModel):
    endpoint: APIEndpoint
    headers: Optional[dict[str, str]] = None
    body: Optional[dict] = None
    timeout: float = Field(default=30.0, ge=1, le=300, description="超时秒数")

    @field_validator('timeout', mode='before')
    @classmethod
    def coerce_timeout(cls, v):
        if isinstance(v, int):
            return float(v)
        if isinstance(v, str):
            try:
                return float(v)
            except ValueError:
                raise ValueError(f'timeout 无法转为数字: {v!r}')
        return v

    @model_validator(mode='after')
    def body_only_for_post_put(self):
        if self.body and self.endpoint.method not in ('POST', 'PUT'):
            raise ValueError(f'{self.endpoint.method} 请求不应包含 body')
        return self

req1 = APIRequest(
    endpoint={"path": "api/v1/users", "method": "GET"},
    timeout="60"
)
print(f"req1: path={req1.endpoint.path}, method={req1.endpoint.method}, timeout={req1.timeout}")

req2 = APIRequest(
    endpoint={"path": "/api/v1/users", "method": "POST"},
    body={"name": "Alice"},
    timeout=30
)
print(f"req2: path={req2.endpoint.path}, method={req2.endpoint.method}, body={req2.body}")

In [ ]:
try:
    APIRequest(
        endpoint={"path": "/api/v1/users", "method": "GET"},
        body={"name": "Alice"}
    )
except ValidationError as e:
    print(f"GET 请求带 body: {e.errors()[0]['msg']}")

try:
    APIRequest(
        endpoint={"path": 123, "method": "GET"},
    )
except ValidationError as e:
    print(f"path 为数字: {e.errors()[0]['msg']}")

---
## 6. 练习

### 练习 1：文件路径校验器
创建一个 `FilePath` 模型，包含 `path` 字段，要求：
- 必须是字符串
- 不能包含 `..`（防止路径穿越）
- 自动去除首尾空格
- 必须以 `/` 开头

In [ ]:
# TODO: 在此编写你的 FilePath 模型
# class FilePath(BaseModel):
#     ...
#     pass

# 测试
# ok = FilePath(path="/home/user/data.txt")
# print(f"合法路径: {ok.path}")

# try:
#     FilePath(path="/home/../etc/passwd")
# except ValidationError as e:
#     print(f"路径穿越被拦截: {e.errors()[0]['msg']}")

### 练习 2：嵌套校验
创建一个 `Order` 模型，包含：
- `items`: 商品列表，每个商品有 `name`、`quantity`（≥1）、`price`（≥0）
- `total`: 总价
- 用 `@model_validator` 校验 `total` 是否等于所有 `quantity * price` 之和

In [ ]:
# TODO: 在此编写你的 Order 模型
# class Item(BaseModel):
#     ...

# class Order(BaseModel):
#     ...

# 测试
# order = Order(
#     items=[
#         {"name": "Apple", "quantity": 2, "price": 5.0},
#         {"name": "Banana", "quantity": 3, "price": 3.0},
#     ],
#     total=19.0
# )
# print(f"合法订单: total={order.total}")

# try:
#     Order(
#         items=[{"name": "Apple", "quantity": 2, "price": 5.0}],
#         total=999.0
#     )
# except ValidationError as e:
#     print(f"总价不匹配: {e.errors()[0]['msg']}")

### 练习 3：动态 Schema
用 `create_model` 动态创建一个日志查询模型，参数根据日志级别动态生成：
- `level`: 只允许传入的日志级别（如 `INFO`, `WARN`, `ERROR`）
- `keyword`: 可选的关键词过滤
- `limit`: 返回条数上限，默认 100

In [ ]:
# TODO: 在此编写你的动态模型生成函数
# def build_log_query_model(levels: list[str]):
#     ...

# 测试
# LogQuery = build_log_query_model(['INFO', 'WARN', 'ERROR'])
# q = LogQuery(level='ERROR', keyword='timeout')
# print(f"合法查询: level={q.level}, keyword={q.keyword}")

# try:
#     LogQuery(level='DEBUG')
# except ValidationError as e:
#     print(f"非法级别: {e.errors()[0]['msg']}")

---
## 总结

| 技术 | 用途 | 关键装饰器/方法 |
|------|------|----------------|
| 字段级校验 | 单字段类型清洗、格式校验 | `@field_validator` |
| 跨字段校验 | 多字段逻辑关系校验 | `@model_validator` |
| 嵌套模型 | 处理复杂 JSON 结构 | Model 嵌套 + 自动递归校验 |
| 动态 Schema | 运行时生成模型、对接 LLM | `create_model` + `model_json_schema()` |

**核心原则**：永远不要信任大模型的输出，在 Pydantic 层做强制拦截和类型清洗。